# 01 · Calibration

This notebook shows the **calibration tool** end to end: wrap a HYPE model, choose an error model, drive both with the `GPU_PSO` optimiser, and run a short `fit` on the bundled Türkheim catchment.

GPU_HYPE calibrates against **two objectives at once**:

1. **reliability / non-exceedance** — how well the spread of the population brackets the observations, and
2. a **streamflow error metric** (here MAE).

Selection keeps a non-dominated front (NSGA-II sorting + crowding distance), so calibration returns a whole **population** of parameter sets on the trade-off between the two — the raw material for the probabilistic forecast in notebooks 02–03.

> This is a deliberately **small, fast** run (few members, few epochs, a short window). A research-grade calibration uses a larger population and more epochs — see [`Trainer.py`](../Trainer.py).

In [ ]:
# --- Bootstrap: run from the repo root ---
import os, sys
from pathlib import Path

def find_repo_root(start=None):
    start = start or Path.cwd()
    for p in [start, *start.parents]:
        if (p / "gpu.py").exists():
            return p
    raise RuntimeError("Could not find the repo root above %s" % start)

REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
%matplotlib inline

from gpu_pso import GPU_PSO
from error.errorOpenCL import Error
from conceptual.HYPE import HYPE
print("Imports OK \u2014 working in", REPO_ROOT)

## 1 · Observations and the calibration window

We calibrate on a short slice (1980–1985) to keep the run quick.

In [ ]:
obs = pd.read_csv("examples/Qobs.txt", sep="\t", header=0,
                  index_col="Date", parse_dates=True)
init_date = pd.to_datetime("1980-01-01")
final_date = pd.to_datetime("1985-12-31")
mask = (obs.index >= init_date) & (obs.index <= final_date)
y = obs.loc[mask, ["50675"]]
print("calibration records:", len(y), "|", y.index[0].date(), "\u2192", y.index[-1].date())

## 2 · The HYPE model wrapper

`HYPE(...)` points the optimiser at a HYPE folder and at the list of `par.txt` parameters to calibrate. It sets up `MultipleRuns` parallel copies of the folder so the population can be evaluated concurrently. `set_simulation_Dates` writes the calibration window into each copy's `info.txt`.

The parameter bounds and whether each is sampled in log-space or as a multiplier are defined inside the `HYPE` class.

In [ ]:
calibration_parameters = ["wcfc", "rrcs1", "ttmp", "cmlt", "preccorr"]

Model = HYPE(
    MultipleRuns=6,                 # parallel HYPE run folders
    records=y.index,
    calibration_parameters=calibration_parameters,
    random=True,                    # random initial population
    normalization=True,
    log=True,                       # default log-space sampling
    HYPEfolder="examples/set7_germany_tuerkheim",
    outfile="results/0050675.txt",  # HYPE basin-output for subbasin 50675
)
Model.set_simulation_Dates(init_date, final_date)
variables = len(Model.parList[0])
print("number of optimisation variables:", variables)

## 3 · The error model

`Error` evaluates the streamflow error with an **OpenCL** kernel (MAE or MSE) on the whole population at once. (A NumPy alternative, `NewErrorModel`, supports MAE/MSE/NSE on the CPU.)

In [ ]:
error_model = Error(errorFunction="MAE")

## 4 · The optimiser

`GPU_PSO` is the Particle-Swarm strategy. The main knobs:

| Argument | Meaning |
|---|---|
| `population` | number of parameter sets (members) evolved together |
| `inertia` | how much of each member's previous velocity is retained |
| `c1`, `c2`, `c3` | pull toward local attractors, global attractors, and random jitter |
| `pBins` | number of local (per-region) attractors along the non-exceedance axis |
| `partial` | fraction of variables perturbed per step |
| `forceNonExceedance` | gentle pressure toward a target reliability |

In [ ]:
opt = GPU_PSO(
    modelObject=Model,
    errorObject=error_model,
    variables=variables,
    population=20,
    inertia=0.2, c1=0.3, c2=0.2, c3=0.001,
    pBins=5, partial=0.3,
    forcePositive=False, transformWeights=False,
    forceNonExceedance=0.01, displayEach=1,
)

## 5 · Calibrate

`fit` runs the optimisation loop. Each epoch it prints a one-line summary: `alpha` (reliability), `xi` (bias), `pi` (sharpness), plus timings. It returns `(result, performance)`.

*(A handful of epochs spawns the HYPE executable many times — expect a minute or two.)*

In [ ]:
# `fit` writes a progress figure every `displayEach` epochs, so give it a save path
# (the figure folder is git-ignored). Without `save=`, fit raises at the end.
os.makedirs("Simulations", exist_ok=True)
save_fig = Path("Simulations/calibration.pplot")

result, performance = opt.fit(y, epochs=5, save=save_fig)
print("\nresult keys:", list(result.keys()))
print("calibrated population (members × variables):", result["parameters"].shape)
print("fitness (members × [non_exceedance, log10 error]):", result["fitness"].shape)

## 6 · The trade-off front

Every calibrated member is a point in (reliability, error) space. The optimiser keeps the non-dominated front — the population we turn into a forecast.

In [ ]:
fit = result["fitness"]
fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(fit[:, 0], fit[:, 1], s=25, edgecolor="k", alpha=0.8)
ax.set_xlabel("Non-exceedance (reliability)")
ax.set_ylabel("log10(MAE)")
ax.set_title("Calibrated population \u2014 objective trade-off")
plt.tight_layout()

## 7 · Save the calibrated model

We free the temporary HYPE run folders, then pickle the whole optimiser so it can be reloaded for prediction. (This file is git-ignored; the repo ships a full pre-trained model used in the next notebooks.)

In [ ]:
opt.modelObject.remove_tmpFiles()
opt.save("examples/my_calibration.pkl")
print("saved \u2192 examples/my_calibration.pkl")

## Next

Continue with [**02 · Training and predicting**](02_training_and_predicting.ipynb), which loads a fully pre-trained model and produces a probabilistic ensemble forecast.